# 01 · RSI / MACD / Bollinger Bands — Origin Scenario: Financial Markets

**Contexto:** Los mismos indicadores del `supply_scenario.ipynb` aplicados al S&P 500 — donde nacieron y donde más se estudian. El objetivo es entender la lógica original antes de trasladarla a Supply Chain.

> Para usar datos reales: `pip install yfinance` → `yf.download('SPY', start='2019-01-01')`

---

## Marco teórico — versión financiera

### RSI sobre precio

$$RSI_t = 100 - \frac{100}{1 + \frac{\bar{G}_{14}}{\bar{L}_{14}}}$$

RSI > 70 en precio → mercado sobrecomprado → probable corrección (~55% de probabilidad a 30 días, Brock et al. 1992).  
**Divergencia bajista:** precio hace nuevo máximo pero RSI no → señal de agotamiento del rally.

### MACD sobre precio

$$MACD = EMA_{12}(P) - EMA_{26}(P) \qquad Signal = EMA_9(MACD)$$

**Golden Cross:** MACD ↑ Signal → momentum alcista → señal de compra.  
**Death Cross:** MACD ↓ Signal → momentum bajista → señal de venta.

### Bollinger Squeeze

$$BW_t = \frac{BB_{upper} - BB_{lower}}{SMA_{20}}$$

Cuando BW cae a mínimos históricos (squeeze) → el mercado consolida → inminente movimiento brusco en cualquier dirección. Usado en estrategias de opciones (volatilidad implícita baja → comprar volatilidad).

### Backtesting — métricas estándar Quant

$$\text{Sharpe} = \frac{\bar{R} - R_f}{\sigma_R} \cdot \sqrt{252} \qquad MDD = \max_t \frac{\text{Peak}_t - P_t}{\text{Peak}_t} \qquad \text{Calmar} = \frac{CAGR}{|MDD|}$$

**Referencias:** Wilder (1978); Brock, Lakonishok & LeBaron (1992) *J. Finance* 47(5); Lo, Mamaysky & Wang (2000) *J. Finance* 55(4).

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    price='#1E293B', sma='#64748B', bb='#0F766E',
    macd='#7C3AED',  sig='#C2410C', rsi='#1D4ED8',
    buy='#15803D',   sell='#B91C1C', fill='#DBEAFE',
    equity='#2563EB', bh='#94A3B8',
)
np.random.seed(2024)
print('✓ OK')

In [ ]:
# ── DATOS ─────────────────────────────────────────────────────────────────────
# Dataset: S&P 500 / SPY ETF — precios diarios de cierre
# Fuente real: Yahoo Finance
#
# Para usar datos reales:
#   import yfinance as yf
#   spy = yf.download('SPY', start='2019-01-01', end='2024-12-31')
#   df  = spy[['Close']].rename(columns={'Close': 'price'})
#
# Simulación reproducible — estadísticos reales SPY 2019-2024:
#   μ diario ≈ +0.052%  ·  σ diaria ≈ 1.10%
#   COVID crash (feb-mar 2020): -34% en 33 días
#   Corrección 2022 (subida Fed): -25% en ~9 meses
#   Precio inicial SPY: $270 (cierre ene 2019)

n = 1510  # ~6 años de días hábiles
dates = pd.bdate_range('2019-01-02', periods=n)

returns = []
for i in range(n):
    if 280 <= i <= 340:          # COVID crash y rebote
        r = np.random.normal(-0.015 if i < 315 else 0.012, 0.035)
    elif 780 <= i <= 1000:       # corrección 2022
        r = np.random.normal(-0.003, 0.020)
    else:
        vol = np.clip(0.5 * 0.011 + 0.5 * abs(returns[-1]) if returns else 0.011,
                      0.005, 0.025)
        r = np.random.normal(0.00052, vol)
    returns.append(r)

price = 270 * np.cumprod(1 + np.array(returns))
df = pd.DataFrame({'price': price}, index=dates)

total_ret = df.price.iloc[-1] / df.price.iloc[0] - 1
ann_vol   = np.std(returns) * np.sqrt(252)
print(f'Serie  : {len(df)} días hábiles ({df.index[0].date()} → {df.index[-1].date()})')
print(f'Rango  : ${df.price.min():.0f} — ${df.price.max():.0f}')
print(f'Retorno total: {total_ret:.1%}')
print(f'Volatilidad anualizada: {ann_vol:.1%}')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
daily_ret = df.price.pct_change().dropna()

stats = {
    'n días'           : (len(df), '~6 años de historia'),
    'Retorno μ diario' : (f'{daily_ret.mean():.4%}', f'anualizado ≈ {daily_ret.mean()*252:.1%}'),
    'Volatilidad diaria': (f'{daily_ret.std():.4%}', f'anualizada ≈ {daily_ret.std()*np.sqrt(252):.1%}'),
    'Sharpe simple'    : (f'{daily_ret.mean()/daily_ret.std()*np.sqrt(252):.3f}', 'sin descontar Rf'),
    'Skewness retornos': (f'{daily_ret.skew():.3f}', 'negativo → cola izquierda pesada (crashes)'),
    'Kurtosis retornos': (f'{daily_ret.kurt():.3f}', '>> 0 → fat tails → riesgo extremo real'),
    'Max retorno diario': (f'{daily_ret.max():.2%}', 'día de mayor alza'),
    'Min retorno diario': (f'{daily_ret.min():.2%}', 'día de mayor caída (crash)'),
    'Días positivos'   : (f'{(daily_ret>0).mean():.1%}', 'win rate del mercado en días individuales'),
}

print(f'{"Métrica":<22} {"Valor":<16} {"Interpretación"}')
print('─' * 72)
for k, (v, interp) in stats.items():
    print(f'{k:<22} {str(v):<16} {interp}')

print()
print('→ Fat tails (kurtosis alta) confirman que la Normal subestima el riesgo.')
print('  Los modelos de momentum como RSI/MACD capturan la asimetría temporal,')
print('  no la distribución de retornos — complementan pero no reemplazan VaR/EVT.')

In [ ]:
# ── EDA 2/2 — Line plot + distribución de retornos ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4),
                          gridspec_kw={'width_ratios': [3, 1]})

ax = axes[0]
ax.plot(df.index, df.price, color=C['price'], lw=1.0, label='SPY precio cierre')
ax.fill_between(df.index, df.price, df.price.min() * 0.9,
                alpha=0.06, color=C['fill'])
ax.set_title('S&P 500 / SPY — precio diario de cierre (simulado 2019-2024)', fontsize=11)
ax.set_ylabel('USD')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.hist(daily_ret * 100, bins=60, color=C['fill'],
         edgecolor=C['price'], linewidth=0.4, orientation='horizontal', density=True)
ax2.axhline(0, color=C['sma'], lw=0.8, ls='--')
ax2.axhline(daily_ret.quantile(0.01) * 100, color=C['sell'], lw=1.0, ls='--', label='VaR 1%')
ax2.set_title('Distribución retornos diarios (%)', fontsize=11)
ax2.set_xlabel('Densidad')
ax2.legend(fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Mini-EDA — S&P 500 / SPY (simulado)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('data/origin_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ Guardado: data/origin_eda.png')

In [ ]:
# ── INDICADORES (mismas funciones, aplicadas a precio) ────────────────────────

def rsi(s, w=14):
    d = s.diff()
    g = d.clip(lower=0).ewm(com=w-1, min_periods=w).mean()
    l = (-d.clip(upper=0)).ewm(com=w-1, min_periods=w).mean()
    return 100 - 100 / (1 + g / l.replace(0, np.nan))

def macd_fn(s, fast=12, slow=26, signal=9):
    m   = s.ewm(span=fast, adjust=False).mean() - s.ewm(span=slow, adjust=False).mean()
    sig = m.ewm(span=signal, adjust=False).mean()
    return pd.DataFrame({'macd': m, 'signal': sig, 'hist': m - sig})

def bollinger(s, w=20, k=2.0):
    sma = s.rolling(w).mean()
    std = s.rolling(w).std()
    up, lo = sma + k*std, sma - k*std
    return pd.DataFrame({
        'sma': sma, 'upper': up, 'lower': lo,
        'bandwidth': (up - lo) / sma,
        'pct_b': (s - lo) / (up - lo)
    })

df['rsi'] = rsi(df.price)
df = pd.concat([df, macd_fn(df.price), bollinger(df.price)], axis=1)

# Cruces MACD → posición long (1) o fuera (0)
df['cross'] = np.where(
    (df.macd > df.signal) & (df.macd.shift(1) <= df.signal.shift(1)),  1,
    np.where(
    (df.macd < df.signal) & (df.macd.shift(1) >= df.signal.shift(1)), -1, 0))

in_pos, positions = False, []
for c in df.cross:
    if   c ==  1: in_pos = True
    elif c == -1: in_pos = False
    positions.append(1 if in_pos else 0)
df['position'] = positions

print(f'Cruces alcistas: {(df.cross== 1).sum()}  |  bajistas: {(df.cross==-1).sum()}')
print(f'Días en posición: {df.position.sum()} de {len(df)} ({df.position.mean():.1%})')

In [ ]:
# ── BACKTESTING ───────────────────────────────────────────────────────────────
df['ret_daily']  = df.price.pct_change()
df['ret_strat']  = df.ret_daily * df.position.shift(1)  # sin lookahead bias
df['equity_strat'] = (1 + df.ret_strat.fillna(0)).cumprod()  * 100
df['equity_bh']    = (1 + df.ret_daily.fillna(0)).cumprod() * 100

def sharpe(r, rf=0.04, p=252):
    ex = r - rf/p
    return np.sqrt(p) * ex.mean() / ex.std()

def mdd(eq):
    return ((eq - eq.cummax()) / eq.cummax()).min()

def cagr(eq, p=252):
    return (eq.iloc[-1] / eq.iloc[0]) ** (p / len(eq)) - 1

rs, rb = df.ret_strat.dropna(), df.ret_daily.dropna()

res = pd.DataFrame({
    'Estrategia MACD': {
        'CAGR'          : cagr(df.equity_strat),
        'Volatilidad'   : rs.std() * np.sqrt(252),
        'Sharpe Ratio'  : sharpe(rs),
        'Max Drawdown'  : mdd(df.equity_strat),
        'Calmar Ratio'  : cagr(df.equity_strat) / abs(mdd(df.equity_strat)),
        'Win Rate'      : (rs[rs != 0] > 0).mean(),
        'Retorno total' : df.equity_strat.iloc[-1]/100 - 1,
    },
    'Buy & Hold': {
        'CAGR'          : cagr(df.equity_bh),
        'Volatilidad'   : rb.std() * np.sqrt(252),
        'Sharpe Ratio'  : sharpe(rb),
        'Max Drawdown'  : mdd(df.equity_bh),
        'Calmar Ratio'  : cagr(df.equity_bh) / abs(mdd(df.equity_bh)),
        'Win Rate'      : (rb > 0).mean(),
        'Retorno total' : df.equity_bh.iloc[-1]/100 - 1,
    }
}).T

print('MÉTRICAS DE BACKTESTING')
print('=' * 55)
pct_cols = ['CAGR', 'Volatilidad', 'Max Drawdown', 'Win Rate', 'Retorno total']
for col in res.columns:
    fmt = '.2%' if col in pct_cols else '.3f'
    print(f"{col:<18}: {res.loc['Estrategia MACD', col]:>8{fmt}}  vs  {res.loc['Buy & Hold', col]:>8{fmt}}")

In [ ]:
# ── DASHBOARD ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 14))
fig.suptitle(
    'RSI · MACD · Bollinger — Financial Markets Dashboard\n'
    'S&P 500 / SPY ETF (simulado 2019-2024)',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(5, 1, hspace=0.07, height_ratios=[3, 1, 1.1, 1.1, 1.5])

# — Panel 1: Precio + BB + señales —
ax1 = fig.add_subplot(gs[0])
ax1.fill_between(df.index, df.upper, df.lower, alpha=0.10, color=C['fill'])
ax1.plot(df.index, df.price, color=C['price'], lw=1.0, label='SPY precio', zorder=3)
ax1.plot(df.index, df.sma,   color=C['sma'], lw=0.8, ls='--', alpha=0.7, label='SMA 20d')
ax1.plot(df.index, df.upper, color=C['bb'], lw=0.6, alpha=0.7)
ax1.plot(df.index, df.lower, color=C['bb'], lw=0.6, alpha=0.7, label='Bollinger ±2σ')
buy_s  = df[df.cross ==  1]
sell_s = df[df.cross == -1]
ax1.scatter(buy_s.index,  buy_s.price,  marker='^', s=60, color=C['buy'],  zorder=5, label='Compra')
ax1.scatter(sell_s.index, sell_s.price, marker='v', s=60, color=C['sell'], zorder=5, label='Venta')
ax1.set_ylabel('USD')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title('Panel 1 — Precio + Bollinger + señales MACD', loc='left', fontsize=10, pad=5)
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticklabels([])

# — Panel 2: Bollinger Bandwidth (squeeze) —
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.fill_between(df.index, df.bandwidth, alpha=0.35, color=C['macd'])
ax2.plot(df.index, df.bandwidth, color=C['macd'], lw=0.8)
ax2.axhline(df.bandwidth.quantile(0.10), color='#CA8A04', lw=0.8, ls='--',
            label='p10 — zona Squeeze')
ax2.set_ylabel('BW')
ax2.legend(fontsize=9)
ax2.set_title('Panel 2 — Bollinger Bandwidth: Squeeze → inminente breakout', loc='left', fontsize=10, pad=4)
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticklabels([])

# — Panel 3: MACD —
ax3 = fig.add_subplot(gs[2], sharex=ax1)

# CORRECCIÓN: Usar df['hist'] entre corchetes
ax3.bar(df.index, df['hist'],
        color=np.where(df['hist'] >= 0, C['buy'], C['sell']),
        alpha=0.5, width=0.8)

ax3.plot(df.index, df.macd,   color=C['macd'], lw=1.2, label='MACD')
ax3.plot(df.index, df.signal, color=C['sig'],  lw=1.0, label='Signal')
ax3.axhline(0, color='#94A3B8', lw=0.7)
ax3.set_ylabel('MACD')
ax3.legend(loc='upper left', fontsize=9)
ax3.set_title('Panel 3 — MACD: momentum del precio', loc='left', fontsize=10, pad=4)
ax3.grid(axis='y', alpha=0.3)
ax3.set_xticklabels([])

# — Panel 4: RSI —
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(df.index, df.rsi, color=C['rsi'], lw=1.2, label='RSI 14d')
ax4.fill_between(df.index, df.rsi, 70, where=df.rsi > 70, alpha=0.15, color=C['sell'])
ax4.fill_between(df.index, df.rsi, 30, where=df.rsi < 30, alpha=0.15, color=C['buy'])
ax4.axhline(70, color=C['sell'], lw=0.8, ls='--', label='Sobrecomprado 70')
ax4.axhline(30, color=C['buy'],  lw=0.8, ls='--', label='Sobrevendido 30')
ax4.axhline(50, color='#94A3B8', lw=0.5, ls=':')
ax4.set_ylim(0, 100)
ax4.set_ylabel('RSI')
ax4.legend(loc='upper left', fontsize=9)
ax4.set_title('Panel 4 — RSI: momentum', loc='left', fontsize=10, pad=4)
ax4.grid(axis='y', alpha=0.3)
ax4.set_xticklabels([])

# — Panel 5: Equity curves —
ax5 = fig.add_subplot(gs[4], sharex=ax1)
ax5.plot(df.index, df.equity_strat, color=C['equity'], lw=1.5, label='Estrategia MACD')
ax5.plot(df.index, df.equity_bh,    color=C['bh'],     lw=1.0, ls='--', label='Buy & Hold')
ax5.axhline(100, color='#94A3B8', lw=0.5, ls=':')
ax5.set_ylabel('Equity (base 100)')
ax5.set_xlabel('Fecha')
ax5.legend(fontsize=9)
ax5.set_title('Panel 5 — Equity Curve: MACD vs Buy & Hold', loc='left', fontsize=10, pad=4)
ax5.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/origin_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ Guardado: data/origin_dashboard.png')

In [ ]:
# ── EXPORTAR ──────────────────────────────────────────────────────────────────
df.to_csv('data/origin_signals_output.csv')
print('✓ data/origin_signals_output.csv')
print('✓ data/origin_eda.png')
print('✓ data/origin_dashboard.png')

## Comparación: Trading (origen) vs Supply Chain (aplicación)

| Dimensión | Trading | Supply Chain |
|-----------|---------|-------------|
| **Serie** | Precio del activo (USD/día) | Demanda del SKU (und/semana) |
| **RSI > 70** | Mercado sobrecomprado → corrección probable | Demanda caliente → riesgo stockout |
| **MACD cruce ↑** | Señal de compra del activo | Señal de aumento de orden |
| **BB_upper** | Precio en extremo alto → posible reversión | Demanda excepcional → reabastecimiento emergencia |
| **Bollinger BW** | Volatilidad implícita baja (Squeeze) | σ de demanda para dimensionar SS dinámico |
| **Métrica éxito** | Sharpe Ratio, Max Drawdown | Fill rate, SS inmovilizado, stockouts |
| **Frecuencia** | Diaria / intradiaria | Semanal / mensual |
| **Parámetros** | RSI 14d, EMA 12/26/9, BB 20d | RSI 14w, EMA 12/26/9, BB 20w |

**Clave de la transferencia:** la matemática es idéntica, cambia la variable (precio → demanda) y la escala temporal (días → semanas). La interpretación económica se adapta: sobrecompra de mercado → sobrecompra de demanda.

**Limitación en Supply que no existe en trading:** en mercados financieros hay short-sellers y arbitrajistas que fuerzan la reversión a la media. En demanda de Supply Chain no. Un RSI alto puede reflejar simplemente un cambio de tendencia permanente. Siempre combinar con el módulo B del EDA (Hurst Exponent) para saber si la serie es mean-reverting o trending antes de asumir corrección.